In [1]:
from datasets import Dataset

In [2]:
data=[
    {
        "instruction": "Match resume with job",
        "input": "Skills: Python, ML | Job: Data Scientist with SQL",
        "output": "Match: 75%, Missing Skills: SQL"
    },
    {
        "instruction": "Match resume with job",
        "input": "Skills: Java, Spring | Job: Backend Developer with Node.js",
        "output": "Match: 60%, Missing Skills: Node.js"
    },
    {
        "instruction": "Match resume with job",
        "input": "Skills: HTML, CSS | Job: Frontend Developer with React",
        "output": "Match: 65%, Missing Skills: React"
    }
]
dataset=Dataset.from_list(data)

In [10]:
print(dataset)

!pip install -U bitsandbytes accelerate transformers peft

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 3
})
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)


# Creating the  4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Loading the model with config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
def format_example(example):
    return {
        "text": f"""
Instruction: {example['instruction']}
Input: {example['input']}
Output: {example['output']}
"""
    }

dataset = dataset.map(format_example)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [5]:
#applying the lora  technique of fine tuning
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [7]:
input_text = """
Instruction: Match resume with job
Input: Skills: Python, ML | Job: AI Engineer with Deep Learning
Output:
"""

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=20)

print(tokenizer.decode(output[0]))

[transformers] Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s> 
Instruction: Match resume with job
Input: Skills: Python, ML | Job: AI Engineer with Deep Learning
Output:
- Python
- Deep Learning
- AI Engineer

Example 2:
Input
